# ДЗ 10. Нейронные операторы: DeepONet и FNO

На лекции вы обучили `DeepONet` аппроксимировать оператор антидифференцирования и `FNO` —
оператор эволюции уравнения Бюргерса. Ключевая идея, отличающая нейронные операторы от PINN
(темы 8-9): PINN решает **одну конкретную** краевую задачу (одно начальное условие, один набор
параметров), тогда как нейронный оператор, один раз обученный на многих примерах, мгновенно
выдаёт решение для **любого нового** входа той же природы — не требуя повторного обучения.

Здесь — новые операторы (не антидифференцирование и не Бюргерс, чтобы не дублировать лекцию),
систематическое исследование архитектурных решений (число сенсоров, размер скрытого
пространства), и прямое сравнение DeepONet/FNO/PINN/классических методов между собой.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42)
np.random.seed(42)

## Уровень 1. Базовый

### Задача 1.1. DeepONet для оператора решения ОДУ

Вместо антидифференцирования (лекция) — оператор решения линейного ОДУ первого порядка:
$$\frac{dy}{dt} = -y(t) + u(t), \qquad y(0) = 0, \qquad t \in [0,1]$$
где $u(t)$ — входная функция (та же "случайная сумма синусоид", что и на лекции), а оператор
$\mathcal{G}: u \mapsto y$ отображает **всю функцию** $u$ в **всю функцию** $y$. Аналитически
это оператор свёртки с экспоненциальным ядром: $y(t) = \int_0^t e^{-(t-s)} u(s)\,ds$.

1. Сгенерируйте датасет: для каждого сэмпла — значения $u$ в $m$ точках-сенсорах (вход branch-
   сети) и значения $y$ в $n\_out$ query-точках (то, что предсказывает trunk-сеть). Истинные
   значения $y$ посчитайте численным интегрированием (например, `numpy.trapezoid` на мелкой
   сетке) — это и есть "эталон", с которым сравнивается DeepONet.
2. Перенесите архитектуру `BranchNet`/`TrunkNet`/`DeepONet` с лекции (branch и trunk — обычные
   MLP, предсказание — скалярное произведение их выходов плюс bias).
3. Обучите и посчитайте относительную $L_2$ ошибку на отложенной выборке.
4. Визуализируйте несколько примеров: входная функция $u(t)$, истинное решение $y(t)$,
   предсказание DeepONet.

In [ ]:
# TODO: DeepONet для оператора решения ОДУ y'=-y+u(t)

### Задача 1.2. Влияние числа сенсоров

Число сенсоров $m$ — сколько точек входной функции $u$ "видит" branch-сеть — определяет, сколько
информации о конкретной входной функции доступно модели.

1. Обучите DeepONet (задача 1.1) с $m \in \{5, 15, 30, 50\}$ при одинаковом бюджете эпох и
   одинаковом $n\_out$.
2. Постройте график относительной $L_2$ ошибки от $m$.
3. Обратите внимание на функции с большим числом гармоник $K$ (более "детальные", быстро
   меняющиеся входы) — ожидается, что при таких входах нехватка сенсоров скажется сильнее, чем
   при гладких, малых $K$. Проверьте эту гипотезу, повторив эксперимент при бОльшем $K$.

In [ ]:
# TODO: сравнение при m = 5,15,30,50, для K=6 и K=15

### Задача 1.3. Влияние размера скрытого пространства $p$

$p$ — размерность выхода branch- и trunk-сетей (сколько "базисных функций" учится комбинировать
DeepONet, аналогично числу мод в разложении Фурье или числу главных компонент в PCA — вспомните
ДЗ 4/5!).

1. Обучите DeepONet (задача 1.1, $m=50$, $K=6$) при $p \in \{2, 5, 15, 40, 100\}$, одинаковый
   бюджет эпох.
2. Постройте график ошибки от $p$ — рано или поздно должно выйти на плато (или даже слегка
   ухудшиться при переобучении).
3. Визуализируйте несколько строк матрицы `trunk(y_query)` (то есть несколько отдельных
   "базисных функций", которые выучил trunk) при большом $p$ — они напоминают что-то знакомое из
   математики (полиномы, синусоиды) или выглядят как нечто более произвольное?

In [ ]:
# TODO: сравнение при p = 2,5,15,40,100, визуализация базисных функций trunk

---
**Что сдавать:** ноутбук с обученными моделями, графиками сравнения и письменными ответами на
вопросы для решённых задач. Большинство задач — от секунд до пары минут на CPU благодаря
использованию аналитических решений там, где это возможно (теплопроводность, ОДУ) вместо дорогих
численных солверов. Задача 3.1 (сравнение с Бюргерсом) требует данных из лекционного
`fno_burgers.ipynb` — при отсутствии сохранённого решателя можно сгенерировать данные заново по
образцу лекции, это не должно занять больше пары минут.